In [1]:
import pandas as pd
import numpy as np
import polars as pl

from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / "data" / "interim"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

CLEAN_DIR.mkdir(exist_ok=True)

_ = pl.Config.set_tbl_rows(20)

In [2]:
# Source-level input only. No CAPM/excess-return logic and no Markowitz reshape here.
# This keeps the original observation level: one stock/date observation.
#
# Index and Treasury tables were dropped from this pipeline: neither is read by any
# downstream modeling notebook (see README.md for the audit).
#
# price/shares_outstanding_thousands are kept alongside the return columns purely to build
# a market-cap series (see cells below). This is NOT used for the return-cleaning logic --
# it exists only so the modeling notebooks can restrict their optimization universe to the
# largest/most liquid names each period, instead of every CRSP common stock (see
# 03_model/analysis_improved_documented.ipynb's methodology note for why: with ~3,300
# stocks and only 60 monthly observations per estimation window, covariance estimation
# becomes computationally very expensive -- Ledoit-Wolf shrinkage cost scales with the
# square of the number of assets -- and the resulting portfolios are hard to distinguish
# from noise).

STOCK_RETURN_COL = "daily_return_with_delisting"

STOCK_COLS = [
    "trade_date",
    "cusip",
    "crsp_security_id_permno",
    STOCK_RETURN_COL,
    "price",
    "shares_outstanding_thousands",
]

In [3]:
# Load the source table.
daily_stock = pd.read_parquet(RAW_DIR / "daily_stock_returns.parquet", columns=STOCK_COLS)

daily_stock["trade_date"] = pd.to_datetime(daily_stock["trade_date"])
daily_stock = daily_stock.sort_values(["cusip", "trade_date"]).reset_index(drop=True)

print(f"daily_stock rows: {len(daily_stock):,}")

daily_stock rows: 8,126,800


In [4]:
# Cleanliness check for the source dataframe.
# Does not reshape the data and does not fill missing returns.
def return_check_summary(df, date_col, key_cols, return_cols, name):
    duplicate_keys = df.duplicated(key_cols).sum()
    summary = {
        "rows": len(df),
        "date_min": df[date_col].min(),
        "date_max": df[date_col].max(),
        "duplicate_keys": int(duplicate_keys),
    }

    for col in return_cols:
        s = df[col]
        summary[f"{col}_null"] = int(s.isna().sum())
        summary[f"{col}_nonfinite"] = int((~np.isfinite(s.dropna())).sum())
        summary[f"{col}_lt_minus_1"] = int((s < -1).sum())
        summary[f"{col}_gt_10"] = int((s > 10).sum())

    print(f"\n{name}")
    for key, value in summary.items():
        if isinstance(value, int):
            print(f"{key}: {value:,}")
        else:
            print(f"{key}: {value}")


return_check_summary(
    daily_stock,
    date_col="trade_date",
    key_cols=["trade_date", "cusip"],
    return_cols=[STOCK_RETURN_COL],
    name="daily_stock",
)


daily_stock
rows: 8,126,800
date_min: 2003-01-02 00:00:00
date_max: 2009-12-31 00:00:00
duplicate_keys: 0
daily_return_with_delisting_null: 2,185
daily_return_with_delisting_nonfinite: 0
daily_return_with_delisting_lt_minus_1: 0
daily_return_with_delisting_gt_10: 4


In [5]:
# Create clean copy while preserving original raw table above.
# Stock cleaning is strict for the selected return: no null, non-finite, or return below -100%.
# The final stock table keeps only date, combined stock identifier, and return.

daily_stock_clean_base = daily_stock.loc[
    daily_stock[STOCK_RETURN_COL].notna()
    & np.isfinite(daily_stock[STOCK_RETURN_COL])
    & (daily_stock[STOCK_RETURN_COL] >= -1)
].copy()

daily_stock_clean_base["cusip"] = daily_stock_clean_base["cusip"].astype("string").str.strip()
daily_stock_clean_base["crsp_security_id_permno"] = (
    daily_stock_clean_base["crsp_security_id_permno"]
    .astype("string")
    .str.strip()
)

daily_stock_clean_base.insert(
    1,
    "cusip_crsp_id",
    daily_stock_clean_base["cusip"] + "_" + daily_stock_clean_base["crsp_security_id_permno"],
)

daily_stock_clean = daily_stock_clean_base[[
    "trade_date",
    "cusip_crsp_id",
    STOCK_RETURN_COL,
]].copy()

print(f"daily_stock_clean rows: {len(daily_stock_clean):,} ({len(daily_stock) - len(daily_stock_clean):,} removed)")
print(f"daily_stock_clean columns: {daily_stock_clean.columns.tolist()}")

daily_stock_clean rows: 8,124,615 (2,185 removed)
daily_stock_clean columns: ['trade_date', 'cusip_crsp_id', 'daily_return_with_delisting']


In [6]:
# Rows with extreme stock returns are retained in daily_stock_clean;
# they are printed here for review rather than silently removed.
stock_return_outliers = daily_stock_clean.loc[
    daily_stock_clean[STOCK_RETURN_COL] > 10,
    ["trade_date", "cusip_crsp_id", STOCK_RETURN_COL],
]

print(stock_return_outliers)


        trade_date   cusip_crsp_id  daily_return_with_delisting
1885202 2003-01-14  20719510_81288                    12.902778
4153337 2009-03-19  49671910_33770                    11.500000
5215788 2009-03-19  65488W10_92012                    16.928572
7754973 2009-03-18  92906L50_84146                    19.000000


# Monthly market capitalization (for universe selection, not for return construction)

Market cap = |price| x shares outstanding (CRSP convention: `price` can be negative,
meaning it is a bid-ask midpoint rather than a closing trade price, hence `abs`).
`shares_outstanding_thousands` is already a point-in-time actual share count, so no
split-adjustment factor is needed here (unlike price series used for return comparisons).

For each stock and month, we keep the market cap as of the **last available trading day
in that month** (a snapshot, not a return -- so no compounding/aggregation function is
needed beyond "last observed value").

This table is saved separately from `monthly_stock_clean` because it answers a different
question (how big is this stock right now) and is only consumed by the modeling notebooks'
universe-selection step, not by the return/frontier math itself.

In [7]:
market_cap_daily = daily_stock_clean_base[[
    "trade_date",
    "cusip_crsp_id",
    "price",
    "shares_outstanding_thousands",
]].copy()

market_cap_daily["market_cap"] = (
    market_cap_daily["price"].abs()
    * market_cap_daily["shares_outstanding_thousands"]
)

# Drop rows where market cap cannot be computed (missing price or shares that day) --
# this does not affect daily_stock_clean or monthly_stock_clean, which are already saved
# independently of this table.
market_cap_daily = market_cap_daily.dropna(subset=["market_cap"])

monthly_market_cap = (
    market_cap_daily
    .sort_values(["cusip_crsp_id", "trade_date"])
    .groupby(
        ["cusip_crsp_id", pd.Grouper(key="trade_date", freq="M")],
        sort=False,
        observed=True,
    )["market_cap"]
    .last()
    .rename("market_cap")
    .reset_index()
    .rename(columns={"trade_date": "month_end_date"})
)

monthly_market_cap.sort_values(
    ["cusip_crsp_id", "month_end_date"],
    inplace=True,
    ignore_index=True,
)

print(f"monthly_market_cap rows: {len(monthly_market_cap):,}")
print(f"monthly_market_cap stocks: {monthly_market_cap['cusip_crsp_id'].nunique():,}")

monthly_market_cap rows: 389,263
monthly_market_cap stocks: 6,832


In [8]:
# Ensure this conversion happens only once
daily_stock_clean["trade_date"] = pd.to_datetime(
    daily_stock_clean["trade_date"]
)

# Temporary gross-return column
daily_stock_clean["_gross_return"] = (
    1.0 + daily_stock_clean[STOCK_RETURN_COL]
)

monthly_stock_clean = (
    daily_stock_clean
    .groupby(
        [
            "cusip_crsp_id",
            pd.Grouper(key="trade_date", freq="M"),
        ],
        sort=False,
        observed=True,
    )["_gross_return"]
    .prod(min_count=1)
    .sub(1.0)
    .rename("monthly_return")
    .reset_index()
)

# Remove the temporary 55-million-row column
daily_stock_clean.drop(columns="_gross_return", inplace=True)

# Sort only the much smaller monthly table
monthly_stock_clean.sort_values(
    ["cusip_crsp_id", "trade_date"],
    inplace=True,
    ignore_index=True,
)

print(f"monthly_stock_clean rows: {len(monthly_stock_clean):,}")

monthly_stock_clean rows: 389,279


In [9]:
# Save clean tables for downstream notebooks.
daily_stock_clean.to_parquet(CLEAN_DIR / "daily_stock_clean.parquet", index=False)
monthly_stock_clean.to_parquet(CLEAN_DIR / "monthly_stock_clean.parquet", index=False)
monthly_market_cap.to_parquet(CLEAN_DIR / "monthly_market_cap.parquet", index=False)

print(f"saved: {CLEAN_DIR / 'daily_stock_clean.parquet'}")
print(f"saved: {CLEAN_DIR / 'monthly_stock_clean.parquet'}")
print(f"saved: {CLEAN_DIR / 'monthly_market_cap.parquet'}")

saved: C:\Users\bigna\Desktop\Quant Projects\Factor Models\claude_final_submission\data\clean\daily_stock_clean.parquet
saved: C:\Users\bigna\Desktop\Quant Projects\Factor Models\claude_final_submission\data\clean\monthly_stock_clean.parquet
saved: C:\Users\bigna\Desktop\Quant Projects\Factor Models\claude_final_submission\data\clean\monthly_market_cap.parquet
